In [ ]:
# ==============================================================================
# Reproducibility Note: To comply with strict reproducibility standards, a fixed
# random seed (random_state=42) is globally enforced across all operations.
# ==============================================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.preprocessing import label_binarize

# Enforce global seed for absolute determinism
np.random.seed(42)

# ==============================================================================
# STEP 1: Load Dataset
# ==============================================================================
# Relative path enables execution on any local machine or environment
data_path = "My dataset with class and without missing values.csv"
df = pd.read_csv(data_path)

# ==============================================================================
# STEP 2: Encode Target Labels
# ==============================================================================
le = LabelEncoder()
y = le.fit_transform(df['mag class'])
class_names = le.classes_.tolist()
print("Classes:", class_names)

# Severe Classes: Strong + Major
severe_indices = [
    class_names.index(c) for c in class_names if c in ['Strong', 'Major']
]

# ==============================================================================
# STEP 3: Define Feature Sets
# ==============================================================================
# PCA BASELINE FEATURES
full_features = [
    'Day', 'Month', 'Year', 'Hour', 'Min', 'Sec',
    'latitude', 'longitude', 'depth', 'magType',
    'gap', 'rms', 'horizontalError', 'depthError','magError','mag'
]

X_full = df[full_features]
# One-hot encode categorical feature
X_full = pd.get_dummies(X_full, columns=['magType'], drop_first=True)

# SHFSF FEATURES
shfsf_features = ['mag', 'magType', 'Year', 'gap', 'magError', 'depth']

X_shfsf = df[shfsf_features]
X_shfsf = pd.get_dummies(X_shfsf, columns=['magType'], drop_first=True)

# ==============================================================================
# STEP 4: Train-Test Split Function
# ==============================================================================
def prepare_data(X_data, y_data):
    X_train, X_test, y_train, y_test = train_test_split(
        X_data, y_data, test_size=0.30, random_state=42, stratify=y_data
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, y_train, y_test

# ==============================================================================
# STEP 5: Evaluation Function
# ==============================================================================
def evaluate_strategy(name, dimension, X_tr, X_te, y_tr, y_te):
    # KNN Classifier (Proxy model for benchmarking representations)
    knn_model = KNeighborsClassifier(n_neighbors=5, metric='euclidean')

    # Train
    knn_model.fit(X_tr, y_tr)

    # Predict
    y_pred = knn_model.predict(X_te)
    y_prob = knn_model.predict_proba(X_te)

    # Metrics
    accuracy = accuracy_score(y_te, y_pred) * 100
    precision = precision_score(y_te, y_pred, average='weighted') * 100
    f1 = f1_score(y_te, y_pred, average='weighted') * 100

    # Recall for severe classes only
    recalls = recall_score(y_te, y_pred, average=None)
    severe_recall = np.mean([recalls[idx] for idx in severe_indices]) * 100

    # ROC-AUC
    y_te_bin = label_binarize(y_te, classes=np.arange(len(class_names)))
    roc_auc = roc_auc_score(y_te_bin, y_prob, average='macro', multi_class='ovr')

    return {
        'Feature Strategy': name,
        'Dimension (d)': dimension,
        'Accuracy (%)': round(accuracy, 2),
        'Precision (%)': round(precision, 2),
        'Recall (Severe Class) (%)': round(severe_recall, 2),
        'F1-Score (%)': round(f1, 2),
        'ROC-AUC': round(roc_auc, 3)
    }

# ==============================================================================
# STEP 6: PCA BASELINE
# ==============================================================================
X_train_f, X_test_f, y_train_f, y_test_f = prepare_data(X_full, y)

# PCA with 9 Components
pca = PCA(n_components=9, random_state=42)
X_train_pca = pca.fit_transform(X_train_f)
X_test_pca = pca.transform(X_test_f)

pca_results = evaluate_strategy(
    name='PCA (Baseline)',
    dimension=9,
    X_tr=X_train_pca,
    X_te=X_test_pca,
    y_tr=y_train_f,
    y_te=y_test_f
)

# ==============================================================================
# STEP 7: SHFSF PROPOSED METHOD
# ==============================================================================
X_train_s, X_test_s, y_train_s, y_test_s = prepare_data(X_shfsf, y)

shfsf_results = evaluate_strategy(
    name='SHFSF (Proposed)',
    dimension=6,
    X_tr=X_train_s,
    X_te=X_test_s,
    y_tr=y_train_s,
    y_te=y_test_s
)

# ==============================================================================
# STEP 8: CREATE FINAL TABLE
# ==============================================================================
results_df = pd.DataFrame([pca_results, shfsf_results])

# ==============================================================================
# STEP 9: DISPLAY RESULTS
# ==============================================================================
print("\n")
print("=" * 90)
print("TABLE 2: Comparative Benchmarking of Feature Representations")
print("=" * 90)
print(results_df.to_string(index=False))

# ==============================================================================
# STEP 10: OPTIONAL EXPORT
# ==============================================================================
results_df.to_csv("Table2_Comparative_Benchmarking.csv", index=False)
print("\nSaved as: Table2_Comparative_Benchmarking.csv")

Classes: ['Major', 'Strong', 'light', 'moderate']


TABLE 2: Comparative Benchmarking of Feature Representations
Feature Strategy  Dimension (d)  Accuracy (%)  Precision (%)  Recall (Severe Class) (%)  F1-Score (%)  ROC-AUC
  PCA (Baseline)              9         94.04          93.67                      21.91         93.63    0.918
SHFSF (Proposed)              6         98.90          98.89                      70.41         98.88    0.981

Saved as: Table2_Comparative_Benchmarking.csv
